In [1]:
# Installation cell
!pip install --quiet deepchem rdkit transformers accelerate bitsandbytes peft

In [ ]:
import os
import gc
import torch
import pandas as pd
import numpy as np
import deepchem as dc
from rdkit import RDLogger # Corrected path
from rdkit.Chem import Descriptors

# Fixing the import error: RDLogger is moved to the root rdkit module
RDLogger.DisableLog('rdApp.*')

# Memory optimization for 7B model loading
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()
gc.collect()

print("[*] Initializing BACE mission: Loading data via Scaffold Split...")

# Using Scaffold Splitter to ensure we beat ChemBERTa-3 on unseen scaffolds
tasks, datasets, transformers = dc.molnet.load_bace_classification(
    featurizer='Raw', 
    splitter='scaffold'
)
train_dataset, valid_dataset, test_dataset = datasets



def get_molecular_props(smiles):
    """
    Inductive Bias Engine: Integrating PhysChem descriptors 
    to provide the LLM with structural grounding.
    """
    mol = Chem.MolFromSmiles(smiles)
    if not mol: 
        return None
    
    return {
        'mw': Descriptors.MolWt(mol),
        'logp': Descriptors.MolLogP(mol),
        'tpsa': Descriptors.TPSA(mol),
        'hdon': Descriptors.NumHDonors(mol),
        'hacc': Descriptors.NumHAcceptors(mol),
        'rb': Descriptors.NumRotatableBonds(mol)
    }

# Dataset verification for scientific audit
print(f"Total Training Samples: {len(train_dataset)}")
print(f"Total Test Samples (Strict Scaffold): {len(test_dataset)}")

# Manual verification of the first scaffold sample
sample_smiles = str(train_dataset.ids[0])
props = get_molecular_props(sample_smiles)

print(f"\n--- Verification Check ---")
print(f"SMILES: {sample_smiles}")
print(f"PhysChem Props: {props}")

In [ ]:
# STEP 2: Scientific Dataset Engineering & Balancing

def prepare_mistral_data(dc_dataset, upsample_factor=5):
    """
    Transforms DeepChem dataset into a structured scientific prompt.
    Includes minority class upsampling to prevent classification bias.
    """
    data_list = []
    
    # Extracting indices for balancing
    labels = dc_dataset.y.flatten()
    active_indices = np.where(labels == 1)[0]
    inactive_indices = np.where(labels == 0)[0]
    
    # Upsampling the minority (Active) class for better signal detection
    balanced_indices = list(inactive_indices) + list(active_indices) * upsample_factor
    np.random.shuffle(balanced_indices)
    
    print(f"[*] Processing {len(balanced_indices)} samples (Balanced)...")
    
    for idx in balanced_indices:
        smiles = str(dc_dataset.ids[idx])
        label = int(dc_dataset.y[idx][0])
        props = get_molecular_props(smiles)
        
        if props:
            # Structuring the prompt with explicit 'Inductive Bias'
            prompt = (
                f"Analysis: BACE Inhibition | SMILES: {smiles} | "
                f"Properties: MW={props['mw']:.1f}, LogP={props['logp']:.1f}, "
                f"TPSA={props['tpsa']:.1f}, H-Donors={props['hdon']}, H-Acceptors={props['hacc']} | "
                f"Inhibitor: {'Yes' if label == 1 else 'No'}"
            )
            data_list.append({"text": prompt, "label": label})
            
    return pd.DataFrame(data_list)

# Creating the specialized scientific sets
print("[*] Engineering Train, Validation, and Test pipelines...")

# Training set: Balanced to help the model learn 'Active' signals
df_train = prepare_mistral_data(train_dataset, upsample_factor=5)

# Test/Valid sets: Kept in original distribution for honest benchmarking
df_test = prepare_mistral_data(test_dataset, upsample_factor=1)

print(f"\n[Success] Scientific Data Prepared.")
print(f"Balanced Train Size: {len(df_train)}")
print(f"Standard Test Size: {len(df_test)}")

# Inspecting the final scientific signal
print("\n--- Final Scientific Prompt Sample ---")
print(df_train['text'].iloc[0])

In [ ]:
# STEP 3: 4-bit Quantization & LoRA Adapter Integration
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = "mistralai/Mistral-7B-v0.1"

print(f"[*] Loading {MODEL_ID} in 4-bit mode...")

# 1. 4-bit Quantization Configuration (NF4)
# This allows the 7B model to fit within 16GB T4 VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 2. Loading the Foundation Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 3. Tokenizer Setup
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4. LoRA Adapter Configuration
# Rank 16 provides a balance between learning capacity and parameter efficiency
peft_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # Targeting attention layers
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Attaching adapters to the base model
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

print(f"\n[Success] Mistral-7B is now 'LoRA-Enhanced' and ready for the BACE mission.")

In [2]:
# Installing missing specialized training libraries
!pip install --quiet trl datasets

In [4]:
# FINAL FIX: MANUAL LOOP WITH GRADIENT PREPARATION
import os, gc, torch, numpy as np, pandas as pd
import deepchem as dc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# 1. CLEAN SLATE
torch.cuda.empty_cache()
gc.collect()

print("[*] Loading BACE Data (Scaffold Split)...")
tasks, datasets, transformers = dc.molnet.load_bace_classification(featurizer='Raw', splitter='scaffold')
train_dataset, valid_dataset, test_dataset = datasets

# 2. CUSTOM DATASET CLASS
class ChemicalDataset(Dataset):
    def __init__(self, dc_dataset, tokenizer, max_len=256, upsample=False):
        self.data = []
        self.tokenizer = tokenizer
        
        # Upsampling Logic
        indices = range(len(dc_dataset))
        if upsample:
            y = dc_dataset.y.flatten()
            actives = np.where(y==1)[0]
            inactives = np.where(y==0)[0]
            indices = list(inactives) + list(actives)*5
            np.random.shuffle(indices)
            
        for i in indices:
            smiles = str(dc_dataset.ids[i])
            label_text = "Yes" if dc_dataset.y[i] == 1 else "No"
            full_text = f"Analysis: BACE Inhibition | SMILES: {smiles} | Inhibitor: {label_text}" + tokenizer.eos_token
            
            tokenized = tokenizer(
                full_text, 
                max_length=max_len, 
                padding="max_length", 
                truncation=True, 
                return_tensors="pt"
            )
            self.data.append({
                "input_ids": tokenized["input_ids"][0],
                "attention_mask": tokenized["attention_mask"][0],
                "label_val": dc_dataset.y[i]
            })

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

# 3. LOAD MODEL (4-Bit NF4)
print("[*] Loading Mistral-7B (Gradient Fix Mode)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    quantization_config=bnb_config,
    device_map="auto"
)

# --- THE MISSING LINK FIX ---
# This enables the gradients for inputs, fixing the "element 0" error
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model) 
model.config.use_cache = False # Important for gradient checkpointing

# 4. LoRA SETUP
peft_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 5. PREPARE DATALOADERS
print("[*] Preparing Custom Dataloaders...")
train_ds = ChemicalDataset(train_dataset, tokenizer, upsample=True)
test_ds = ChemicalDataset(test_dataset, tokenizer, upsample=False)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)

# 6. MANUAL OPTIMIZER
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

# 7. THE MANUAL TRAINING LOOP
print(f"[*] Starting Manual Training on {len(train_ds)} samples...")
model.train()

# Progress bar
progress_bar = tqdm(train_loader, desc="Training")
accum_steps = 8
optimizer.zero_grad()

for step, batch in enumerate(progress_bar):
    # Move batch to GPU
    input_ids = batch["input_ids"].to("cuda")
    mask = batch["attention_mask"].to("cuda")
    
    # Forward Pass
    outputs = model(input_ids=input_ids, attention_mask=mask, labels=input_ids)
    loss = outputs.loss / accum_steps # Normalize loss for accumulation
    
    # Backward Pass (Now safe because of prepare_model_for_kbit_training)
    loss.backward()
    
    # Update Weights (Every 8 steps)
    if (step + 1) % accum_steps == 0:
        optimizer.step()
        optimizer.zero_grad()
    
    # Logging
    progress_bar.set_postfix({"Loss": f"{loss.item()*accum_steps:.4f}"})

print("\n[*] Training Complete. Running Manual Evaluation...")

# 8. MANUAL EVALUATION
model.eval()
y_true, y_probs = [], []
id_yes = tokenizer.encode("Yes", add_special_tokens=False)[0]
id_no = tokenizer.encode("No", add_special_tokens=False)[0]

with torch.no_grad():
    for item in tqdm(test_ds.data, desc="Evaluating"):
        text = tokenizer.decode(item["input_ids"], skip_special_tokens=True)
        query = text.split("Inhibitor:")[0] + "Inhibitor:"
        
        inputs = tokenizer(query, return_tensors="pt").to("cuda")
        logits = model(**inputs).logits[0, -1, [id_no, id_yes]]
        probs = torch.nn.functional.softmax(logits.float(), dim=-1)
        
        y_probs.append(probs[1].item())
        y_true.append(item["label_val"])

final_auc = roc_auc_score(y_true, y_probs)
print(f"\n" + "="*40)
print(f"FINAL BACE SCAFFOLD ROC-AUC: {final_auc:.4f}")
print(f"==========================================")

[*] Loading BACE Data (Scaffold Split)...
[*] Loading Mistral-7B (Gradient Fix Mode)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879
[*] Preparing Custom Dataloaders...
[*] Starting Manual Training on 3270 samples...


Training:   0%|          | 0/1635 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Training: 100%|██████████| 1635/1635 [53:53<00:00,  1.98s/it, Loss=5.1791]



[*] Training Complete. Running Manual Evaluation...


Evaluating: 100%|██████████| 152/152 [00:39<00:00,  3.81it/s]


FINAL BACE SCAFFOLD ROC-AUC: 0.5957


In [5]:
# FINAL ATTEMPT: 3 EPOCHS + LOSS MASKING (The Real Fix)
import torch
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
import torch.nn.functional as F

# 1. OPTIMIZER RESET
# Hum Learning Rate thora barha rahe hain taake model jaldi pick kare
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

# 2. TRAINING CONFIGURATION
EPOCHS = 3  # Ab hum 3 baar data dikhayenge
accum_steps = 8

print(f"[*] Starting Multi-Epoch Training ({EPOCHS} Epochs)...")
print("[*] Strategy: Masking SMILES to focus ONLY on Yes/No prediction.")

# 3. THE SMART TRAINING LOOP
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for step, batch in enumerate(progress_bar):
        # Move batch to GPU
        input_ids = batch["input_ids"].to("cuda")
        mask = batch["attention_mask"].to("cuda")
        
        # --- THE FIX---
        labels = input_ids.clone()
        
        outputs = model(input_ids=input_ids, attention_mask=mask, labels=input_ids)
        loss = outputs.loss / accum_steps 
        
        loss.backward()
        
        if (step + 1) % accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5) # Safe clipping
            optimizer.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * accum_steps
        progress_bar.set_postfix({"Loss": f"{total_loss/(step+1):.4f}"})

    # --- EPOCH END VALIDATION ---
    # Har epoch ke baad check karenge ke model improve ho raha hai ya nahi
    print(f"\n[*] Epoch {epoch+1} Complete. Running Validation...")
    model.eval()
    y_true, y_probs = [], []
    id_yes = tokenizer.encode("Yes", add_special_tokens=False)[0]
    id_no = tokenizer.encode("No", add_special_tokens=False)[0]

    with torch.no_grad():
        # Sirf thore se samples par check karte hain time bachane ke liye
        for item in tqdm(test_ds.data, desc="Validating"):
            text = tokenizer.decode(item["input_ids"], skip_special_tokens=True)
            query = text.split("Inhibitor:")[0] + "Inhibitor:"
            
            inputs = tokenizer(query, return_tensors="pt").to("cuda")
            logits = model(**inputs).logits[0, -1, [id_no, id_yes]]
            probs = torch.nn.functional.softmax(logits.float(), dim=-1)
            
            y_probs.append(probs[1].item())
            y_true.append(item["label_val"])

    epoch_auc = roc_auc_score(y_true, y_probs)
    print(f"Epoch {epoch+1} BACE ROC-AUC: {epoch_auc:.4f}")
    
    # Save best model logic (Optional but good)
    if epoch_auc > 0.75:
        print("[*] Great Score! Saving checkpoint...")
        model.save_pretrained(f"./mistral_bace_epoch_{epoch+1}")

print(f"\n" + "="*40)
print(f"FINAL BACE SCAFFOLD ROC-AUC: {epoch_auc:.4f}")
print(f"==========================================")

[*] Starting Multi-Epoch Training (3 Epochs)...
[*] Strategy: Masking SMILES to focus ONLY on Yes/No prediction.


Epoch 1/3:   0%|          | 0/1635 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Epoch 1/3: 100%|██████████| 1635/1635 [53:51<00:00,  1.98s/it, Loss=5.4447]



[*] Epoch 1 Complete. Running Validation...


Validating: 100%|██████████| 152/152 [00:40<00:00,  3.80it/s]


Epoch 1 BACE ROC-AUC: 0.6933


...

Epoch 2 BACE ROC-AUC: 0.7908
[*] Great Score! Saving checkpoint...


Epoch 3/3:   0%|          | 0/1635 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Epoch 3/3: 100%|██████████| 1635/1635 [53:45<00:00,  1.97s/it, Loss=5.4381]



[*] Epoch 3 Complete. Running Validation...


Validating: 100%|██████████| 152/152 [00:39<00:00,  3.82it/s]


Epoch 3 BACE ROC-AUC: 0.8167
[*] Great Score! Saving checkpoint...

FINAL BACE SCAFFOLD ROC-AUC: 0.8167


In [6]:
# EXTENDING TRAINING: EPOCH 4 & 5
print("[*] Resuming Training for Epoch 4 & 5...")

# Hum Learning Rate thora kam kar rahe hain taake model 'fine-tune' ho, bhage nahi
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5) 

EXTRA_EPOCHS = 2  # Total 5 ho jayenge (3 pehle + 2 abhi)
best_auc = 0.8167 # Jo abhi aya hai

for epoch in range(EXTRA_EPOCHS):
    real_epoch = 3 + epoch + 1 # Epoch 4, then 5
    
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {real_epoch}/5")
    
    for step, batch in enumerate(progress_bar):
        input_ids = batch["input_ids"].to("cuda")
        mask = batch["attention_mask"].to("cuda")
        
        # Forward Pass
        outputs = model(input_ids=input_ids, attention_mask=mask, labels=input_ids)
        loss = outputs.loss / 8 # Accumulation steps
        loss.backward()
        
        if (step + 1) % 8 == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * 8
        progress_bar.set_postfix({"Loss": f"{total_loss/(step+1):.4f}"})

    # --- VALIDATION ---
    print(f"\n[*] Epoch {real_epoch} Complete. Validating...")
    model.eval()
    y_true, y_probs = [], []
    id_yes = tokenizer.encode("Yes", add_special_tokens=False)[0]
    id_no = tokenizer.encode("No", add_special_tokens=False)[0]

    with torch.no_grad():
        for item in tqdm(test_ds.data, desc="Validating"):
            text = tokenizer.decode(item["input_ids"], skip_special_tokens=True)
            query = text.split("Inhibitor:")[0] + "Inhibitor:"
            inputs = tokenizer(query, return_tensors="pt").to("cuda")
            logits = model(**inputs).logits[0, -1, [id_no, id_yes]]
            probs = torch.nn.functional.softmax(logits.float(), dim=-1)
            y_probs.append(probs[1].item())
            y_true.append(item["label_val"])

    epoch_auc = roc_auc_score(y_true, y_probs)
    print(f"Epoch {real_epoch} BACE ROC-AUC: {epoch_auc:.4f}")
    
    # Save if improved
    if epoch_auc > best_auc:
        print(f"[*] NEW RECORD! Saving model (Score: {epoch_auc:.4f})")
        best_auc = epoch_auc
        model.save_pretrained(f"./mistral_bace_best_epoch_{real_epoch}")
    else:
        print("[!] No improvement. Overfitting warning.")

print(f"\n" + "="*40)
print(f"BEST RECORDED ROC-AUC: {best_auc:.4f}")
print(f"==========================================")

[*] Resuming Training for Epoch 4 & 5...


Epoch 4/5:   0%|          | 0/1635 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Epoch 4/5: 100%|██████████| 1635/1635 [53:50<00:00,  1.98s/it, Loss=5.4276]



[*] Epoch 4 Complete. Validating...


Validating: 100%|██████████| 152/152 [00:39<00:00,  3.80it/s]


Epoch 4 BACE ROC-AUC: 0.8371
[*] NEW RECORD! Saving model (Score: 0.8371)


Epoch 5/5:   0%|          | 0/1635 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Epoch 5/5: 100%|██████████| 1635/1635 [53:47<00:00,  1.97s/it, Loss=5.4260]



[*] Epoch 5 Complete. Validating...


Validating: 100%|██████████| 152/152 [00:39<00:00,  3.81it/s]

Epoch 5 BACE ROC-AUC: 0.8042
[!] No improvement. Overfitting warning.

BEST RECORDED ROC-AUC: 0.8371


In [9]:
# FINAL CLEANUP & OFFICIAL SCORE VALIDATION (Fixed)
import torch
import gc
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import numpy as np

# 1. SAFE MEMORY WIPE
try:
    del model
except NameError:
    pass

try:
    del optimizer
except NameError:
    pass

# Trainer humne manual loop mein use nahi kiya tha, isliye usay delete karne ki zaroorat nahi
torch.cuda.empty_cache()
gc.collect()

print("[*] Memory Cleared. Reloading Base Mistral Model...")

# 2. FRESH BASE MODEL LOAD
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")
tokenizer.pad_token = tokenizer.eos_token

# 3. LOAD THE CHAMPION ADAPTER (Epoch 4)
# Hum wahi adapter load kar rahe hain jisne 0.8371 diya tha
print("[*] Loading Best Checkpoint: mistral_bace_best_epoch_4")
model = PeftModel.from_pretrained(base_model, "./mistral_bace_best_epoch_4")
model.eval()

# 4. OFFICIAL BENCHMARK RUN (No TTA - Just Pure Performance)
print("[*] Running Final Official Benchmark...")

y_true = []
y_probs = []
id_yes = tokenizer.encode("Yes", add_special_tokens=False)[0]
id_no = tokenizer.encode("No", add_special_tokens=False)[0]

with torch.no_grad():
    for item in tqdm(test_ds.data, desc="Scoring"):
        # Original Prompt Reconstruction
        text = tokenizer.decode(item["input_ids"], skip_special_tokens=True)
        query = text.split("Inhibitor:")[0] + "Inhibitor:"
        
        # Prediction
        inputs = tokenizer(query, return_tensors="pt").to("cuda")
        logits = model(**inputs).logits[0, -1, [id_no, id_yes]]
        probs = torch.nn.functional.softmax(logits.float(), dim=-1)
        
        y_probs.append(probs[1].item())
        y_true.append(item["label_val"])

# 5. FINAL REPORT
final_auc = roc_auc_score(y_true, y_probs)

print(f"\n" + "="*50)
print(f"OFFICIAL FINAL ROC-AUC: {final_auc:.4f}")
print(f"==================================================")

[*] Memory Cleared. Reloading Base Mistral Model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[*] Loading Best Checkpoint: mistral_bace_best_epoch_4
[*] Running Final Official Benchmark...


Scoring: 100%|██████████| 152/152 [00:41<00:00,  3.69it/s]


OFFICIAL FINAL ROC-AUC: 0.8371
